# Pecah Excel besar untuk diunggah

Gunakan runtime **CPU** dan jalankan dari atas. Pilih satu atau beberapa file Excel dari komputer, misalnya hasil E1 dan E2. Notebook memecah tiap file menjadi bagian maksimal **30 MiB** dan memverifikasi bahwa gabungannya identik dengan file asli.

Setelah selesai, klik setiap tombol unduh. Unggah semua file **.part001, .part002, dan seterusnya**, beserta **.manifest.json**, ke chat. File Excel 91 MB akan menjadi sekitar 3–4 bagian. Setiap bagian merupakan potongan biner untuk digabung kembali.

File diproses di runtime Colab yang sedang dibuka. Notebook ini bisa digunakan dari sesi baru.


## 1 · Upload Excel dari komputer

In [ ]:
from pathlib import Path
from tempfile import mkdtemp
from google.colab import files
from IPython.display import display
import ipywidgets as widgets

uploaded = files.upload()
if not uploaded:
    raise ValueError("Belum ada file yang dipilih.")

source_folder = Path(mkdtemp(prefix="excel_asli_"))
source_files = []
for name, content in uploaded.items():
    if Path(name).suffix.lower() not in {".xlsx", ".xls", ".xlsm", ".xlsb"}:
        raise ValueError("Pilih file Excel: " + name)
    path = source_folder / Path(name).name
    path.write_bytes(content)
    source_files.append(path)
    print(f"{path.name} — {path.stat().st_size / 1024**2:.2f} MiB")

del uploaded


## 2 · Pecah dan verifikasi file

In [ ]:
"""Split an uploaded Excel file into verified pieces suitable for attachment upload."""

import hashlib
import json
from pathlib import Path


def split_excel_file(source, destination, part_size=30 * 1024 * 1024):
    """Return a manifest after checking that concatenated pieces match the source."""
    source = Path(source)
    destination = Path(destination)
    if not source.is_file() or source.stat().st_size == 0:
        raise ValueError("Pilih file Excel yang ada dan berisi data.")
    if not 1 <= part_size <= 30 * 1024 * 1024:
        raise ValueError("Ukuran bagian harus antara 1 byte dan 30 MiB.")
    destination.mkdir(parents=True, exist_ok=True)
    source_hash = hashlib.sha256()
    parts = []

    with source.open("rb") as stream:
        number = 1
        while chunk := stream.read(part_size):
            name = f"{source.name}.part{number:03d}"
            path = destination / name
            with path.open("xb") as target:
                target.write(chunk)
            source_hash.update(chunk)
            parts.append({
                "name": name,
                "size_bytes": len(chunk),
                "sha256": hashlib.sha256(chunk).hexdigest(),
            })
            number += 1

    reconstructed_hash = hashlib.sha256()
    total_bytes = 0
    for part in parts:
        part_hash = hashlib.sha256()
        size = 0
        with (destination / part["name"]).open("rb") as stream:
            while block := stream.read(1024 * 1024):
                reconstructed_hash.update(block)
                part_hash.update(block)
                size += len(block)
        if size != part["size_bytes"] or part_hash.hexdigest() != part["sha256"]:
            raise IOError("Verifikasi bagian gagal: " + part["name"])
        total_bytes += size

    if total_bytes != source.stat().st_size or reconstructed_hash.hexdigest() != source_hash.hexdigest():
        raise IOError("Gabungan pecahan berbeda dari file asli.")

    manifest = {
        "original_name": source.name,
        "size_bytes": total_bytes,
        "sha256": source_hash.hexdigest(),
        "part_size_bytes": part_size,
        "parts": parts,
    }
    manifest_path = destination / f"{source.name}.manifest.json"
    with manifest_path.open("x", encoding="utf-8") as target:
        json.dump(manifest, target, ensure_ascii=False, indent=2)
    return manifest, manifest_path


In [ ]:
result_files = []
for source in source_files:
    destination = Path(mkdtemp(prefix="pecahan_excel_"))
    manifest, manifest_path = split_excel_file(source, destination)
    print(f"\n{source.name}: {len(manifest['parts'])} bagian, verifikasi berhasil.")
    for part in manifest["parts"]:
        result_files.append(destination / part["name"])
        print(f"  {part['name']} — {part['size_bytes'] / 1024**2:.2f} MiB")
    result_files.append(manifest_path)
    print("SHA-256 asli:", manifest["sha256"])


## 3 · Unduh semua bagian dan manifest

Klik tiap tombol satu per satu. Tunggu unduhan selesai sebelum menekan tombol berikutnya. Tombol bisa diklik ulang kalau unduhan terputus.

In [ ]:
def make_download_button(path):
    button = widgets.Button(
        description="Unduh " + path.name,
        layout=widgets.Layout(width="100%"),
    )
    def download_clicked(_):
        files.download(str(path))
    button.on_click(download_clicked)
    return button

display(widgets.VBox([make_download_button(path) for path in result_files]))
print("Unggah semua bagian beserta manifest ke chat untuk digabung dan dianalisis.")
